In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import matplotlib
import seaborn as sns
import copy
import pickle
import warnings
warnings.filterwarnings('ignore')
import shap
from models import *
from utilities import *

2026-01-14 15:02:27.493255: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[INFO] No GPU found, using CPU.


In [3]:
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42 
matplotlib.rc('font', family='sans-serif') 
matplotlib.rc('font', serif='Helvetica Neue') 

In [4]:
##########################################################################################################

In [5]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("Eager mode:", tf.executing_eagerly())
print("GPUs:", tf.config.list_physical_devices('GPU'))

TF version: 2.15.0
Eager mode: True
GPUs: []


In [6]:
# shap.explainers._deep.deep_tf.op_handlers["AddV2"] = shap.explainers._deep.deep_tf.passthrough 
# shap.explainers._deep.deep_tf.op_handlers["FusedBatchNormV3"] = shap.explainers._deep.deep_tf.passthrough

In [7]:
################### data import ###############

In [8]:
data_dir = '../results/data/'
data_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_dy_filter.pkl')
ba_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_ba_filter.pkl')

In [9]:
filtered = data_copy.groupby("SCCRIP_ID").filter(lambda x: len(x) >= 5)

In [10]:
data_copy.shape, ba_copy.shape, filtered.shape, len(data_copy.SCCRIP_ID.unique())

((34927, 789), (1338, 24), (34080, 789), 1267)

In [11]:
len(filtered.SCCRIP_ID.unique())

963

In [12]:
len(filtered[filtered['HU_status'] == 0]), len(filtered[filtered['HU_status'] != 0])

(10076, 24004)

In [13]:
len(filtered)/963

35.389408099688474

In [14]:
ba_copy = ba_copy[ba_copy['SCCRIP_ID'].isin(filtered.SCCRIP_ID.unique())]
ba_copy = ba_copy.drop_duplicates()
len(ba_copy.SCCRIP_ID.unique())

963

In [15]:
with open ('../results/data/mapping_dict_background.pkl', 'rb') as f:
    map_dict = pickle.load(f)

In [16]:
map_dict.keys()

dict_keys(['Cohort_21Freeze_ctxn', 'Cohort_21Freeze_voxelotor', 'Gender_demographic', 'Cohort_21Freeze_demographic', 'Race_demographic', 'Ethnicity_demographic', 'Comment_l_gluatmine', 'Cohort_21Freeze_l_gluatmine', 'Comment_crizanzumab', 'Cohort_21Freeze_crizanzumab', 'Genotype_demographic'])

In [17]:
mapping_rev = {v: k for k, v in map_dict['Gender_demographic'].items()}

ba_copy['gender'] = ba_copy['Gender_demographic'].map(mapping_rev)

In [18]:
mapping_rev = {v: k for k, v in map_dict[ 'Genotype_demographic'].items()}

ba_copy['genotype'] = ba_copy['Genotype_demographic'].map(mapping_rev)

In [19]:
g = filtered.groupby('SCCRIP_ID')
y = []
ids = []
for k in g.groups.keys():
    df = g.get_group(k)

    delta = df["date"].max() - df["date"].min()
    years = round(delta.days / 365, 1)
    y.append(years)
    ids.append(k)

In [20]:
df_y = pd.DataFrame({'SCCRIP_ID':ids, 'span':y})

In [21]:
delta = {}
all_delta = []
for k in g.groups.keys():
    df = g.get_group(k)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    
    df["next_date"] = df["date"].shift(-1)
    df["delta"] = df["next_date"] - df["date"]
    
    # convert to months (average month length)
    df["months"] = df["delta"].dt.total_seconds() / (30.44 * 24 * 3600)
    df["months"] = df["months"].round(2)
    
    delta[k] = list(df["months"])
    all_delta.extend(list(df["months"]))

In [22]:
a = np.array(all_delta)
np.nanmin(a),np.nanmax(a), np.nanmedian(a)

(0.03, 209.17, 1.84)

In [23]:
filtered['Ages'].median(), filtered['Ages'].max(), filtered['Ages'].min()

(10.480492813141684, 62.882956878850095, 0.0)

In [24]:
ba_copy.groupby('gender').count()

,SCCRIP_ID,Age_at_CTXN_ctxn,CTXN_Duration1_ctxn,CTXN_Duration2_ctxn,CTXN_Duration3_ctxn,BMT_othertreatment_summary,L_Glut_othertreatment_summary,Voxelotor_othertreatment_summary,Crizanzumab_othertreatment_summary,Other_treat_othertreatment_summary,...,Ethnicity_demographic,Zip_code_demographic,LG_Start_DT_l_gluatmine,LG_END_DT_l_gluatmine,Age_LGluatmine_l_gluatmine,Crizan_Start_DT_crizanzumab,Crizan_END_DT_crizanzumab,Age_Crizanzumab_crizanzumab,Genotype_demographic,genotype
gender,,,,,,,,,,,,,,,,,,,,,
b'Female',483,104,102,19,0,479,479,479,479,479,...,483,483,4,2,4,8,6,8,483,483
b'Male',480,113,113,18,3,477,477,477,477,477,...,480,480,5,2,5,6,2,6,480,480


In [25]:
483/963, 480/963

(0.5015576323987538, 0.4984423676012461)

In [26]:
df_dynamic = pd.read_pickle(data_dir + 'dynamic.pkl')
df_background = pd.read_pickle(data_dir + 'background.pkl')

In [27]:
df_ba = df_background[df_background.columns.intersection(ba_copy.columns)]

In [28]:
df_ba.dtypes

SCCRIP_ID                                     object
Age_at_CTXN_ctxn                             float64
CTXN_Duration1_ctxn                          float64
CTXN_Duration2_ctxn                          float64
CTXN_Duration3_ctxn                          float64
BMT_othertreatment_summary                   float64
L_Glut_othertreatment_summary                float64
Voxelotor_othertreatment_summary             float64
Crizanzumab_othertreatment_summary           float64
Other_treat_othertreatment_summary           float64
Gender_demographic                            object
Genotype_demographic                          object
Age_2021_demographic                         float64
Cohort_21Freeze_demographic                   object
Follow_interval_demographic                  float64
Race_demographic                              object
Ethnicity_demographic                         object
Zip_code_demographic                         float64
LG_Start_DT_l_gluatmine               datetime

In [29]:
df_dy = df_dynamic[df_dynamic.columns.intersection(filtered.columns)]

In [30]:
df_dy.shape

(5447722, 786)

In [31]:
df_dy.dtypes

SCCRIP_ID                           object
BloodChem_TM_bloodchem             float64
BUN_bloodchem                      float64
Creatinine_bloodchem               float64
Glucose_Lvl_bloodchem              float64
                                    ...   
WT_weights_and_measures            float64
BMI_weights_and_measures           float64
BMI_CAL_weights_and_measures       float64
BSA_weights_and_measures           float64
HeadCircum_weights_and_measures    float64
Length: 786, dtype: object

In [32]:
from pandas.api.types import (
    is_numeric_dtype,
    is_string_dtype,
    is_datetime64_any_dtype
)

ba_type_df = pd.DataFrame({
    "column": df_ba.columns,
    "type": [
        "numeric" if is_numeric_dtype(df_ba[c]) else
        "datetime" if is_datetime64_any_dtype(df_ba[c]) else
        "character" 
        for c in df_ba.columns
    ]
})

In [33]:
ba_type_df

,column,type
0,SCCRIP_ID,character
1,Age_at_CTXN_ctxn,numeric
2,CTXN_Duration1_ctxn,numeric
3,CTXN_Duration2_ctxn,numeric
4,CTXN_Duration3_ctxn,numeric
5,BMT_othertreatment_summary,numeric
6,L_Glut_othertreatment_summary,numeric
7,Voxelotor_othertreatment_summary,numeric
8,Crizanzumab_othertreatment_summary,numeric
9,Other_treat_othertreatment_summary,numeric


In [34]:
dy_type_df = pd.DataFrame({
    "column": df_dy.columns,
    "type": [
        "numeric" if is_numeric_dtype(df_dy[c]) else
        "datetime" if is_datetime64_any_dtype(df_dy[c]) else
        "character" 
        for c in df_dy.columns
    ]
})

In [35]:
dy_type_df

,column,type
0,SCCRIP_ID,character
1,BloodChem_TM_bloodchem,numeric
2,BUN_bloodchem,numeric
3,Creatinine_bloodchem,numeric
4,Glucose_Lvl_bloodchem,numeric
...,...,...
781,WT_weights_and_measures,numeric
782,BMI_weights_and_measures,numeric
783,BMI_CAL_weights_and_measures,numeric
784,BSA_weights_and_measures,numeric


In [36]:
with pd.ExcelWriter(data_dir + "Table SI.xlsx", engine="xlsxwriter") as writer:
    dy_type_df.to_excel(writer, sheet_name="dynamic features", index=False)
    ba_type_df.to_excel(writer, sheet_name="static features", index=False)